# Transformers Lab — Pretrained Encoder–Decoder and Decoder-Only Models

**Course:** CS F407 — Artificial Intelligence · **Lab:** Transformers

---

## Scope

This lab has no separate handout PDF. Working from the three tasks it covers:

1. **Translation** with an encoder–decoder model (MarianMT, `opus-mt-en-fr`);
2. **Next-token prediction / sentence completion** with a decoder-only causal
   model (GPT-2);
3. **Sentiment analysis** with an encoder-only classifier (DistilBERT fine-tuned
   on SST-2).

All three run on CPU with pretrained weights from the Hugging Face Hub. Nothing
is trained here — the point is to use models that already exist and understand
*what kind of object each one is*.

## How this connects to the other labs

This is the third time this course has modelled $P(X_t \mid X_{<t})$. The
Bayesian Networks lab built that conditional as an explicit count table; the
Neural Models lab built a tiny network and derived its gradients by hand. GPT-2
is the same conditional distribution again — same chain-rule factorisation, same
softmax over a vocabulary, same $p - y$ gradient during its training — estimated
by a 124-million-parameter transformer instead.

So rather than treating the three tasks as unrelated demos, this notebook uses
them to make one architectural point: **the three model families differ in what
attends to what**, and that single difference determines what each can be used
for. The notebook measures the difference rather than asserting it —
parameter-count breakdowns, an explicit check that the decoder is causally
masked, and next-token distributions inspected directly.

In [1]:
"""Environment. CPU only; all weights are downloaded pretrained."""
import os
import textwrap
import warnings

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import torch
import transformers
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          AutoModelForCausalLM, AutoModelForSequenceClassification,
                          logging as hf_logging)

hf_logging.set_verbosity_error()      # keep the notebook output readable

torch.set_num_threads(2)
DEVICE = torch.device("cpu")
torch.manual_seed(407)

print(f"transformers {transformers.__version__} | torch {torch.__version__}")
print(f"device: {DEVICE}")

def human(n):
    for unit in ("", "K", "M", "B"):
        if abs(n) < 1000:
            return f"{n:.1f}{unit}"
        n /= 1000
    return f"{n:.1f}T"

def wrap(text, width=78, indent="    "):
    return textwrap.fill(str(text), width=width,
                         initial_indent=indent, subsequent_indent=indent)

transformers 5.17.0 | torch 2.7.1
device: cpu


---

## Part 1 — Translation with an encoder–decoder model

### Why this task needs *two* stacks

Translation is a **sequence-to-sequence** problem, and its defining awkwardness
is that the output is not an edit of the input: French word order, length and
morphology all differ from English, so there is no position-by-position
correspondence to exploit.

That is what the encoder–decoder split is for:

- the **encoder** reads the entire English sentence with *bidirectional*
  self-attention — every token attends to every other, both directions, because
  the whole input is available at once;
- the **decoder** generates French left to right with *causal* self-attention
  (it must not see future tokens it has yet to produce) **plus cross-attention**
  into the encoder's output, which is how each French token gets to consult the
  whole English sentence.

Cross-attention is the component that does the actual aligning, and it is what
neither of the other two architectures in this notebook has.

`Helsinki-NLP/opus-mt-en-fr` is a MarianMT model — a compact transformer trained
on the OPUS parallel corpora.

In [2]:
MT_NAME = "Helsinki-NLP/opus-mt-en-fr"
mt_tokenizer = AutoTokenizer.from_pretrained(MT_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(DEVICE).eval()

cfg = mt_model.config
total = sum(p.numel() for p in mt_model.parameters())
shared_embed = mt_model.model.shared.weight.numel()
# NOTE: the source/target embedding matrix is SHARED, so it is reachable from
# both stacks. Counting it in each would double-count it, so report the
# transformer layers separately from the embedding table.
enc_layers = sum(p.numel() for p in mt_model.model.encoder.parameters()) - shared_embed
dec_layers = sum(p.numel() for p in mt_model.model.decoder.parameters()) - shared_embed

print(f"Model : {MT_NAME}")
print(f"  total parameters      : {total:,} ({human(total)})")
print(f"  shared embedding table: {shared_embed:,} ({100*shared_embed/total:.0f}%)")
print(f"  encoder layers only   : {enc_layers:,} ({100*enc_layers/total:.0f}%)")
print(f"  decoder layers only   : {dec_layers:,} ({100*dec_layers/total:.0f}%)")
print(f"  encoder layers        : {cfg.encoder_layers}")
print(f"  decoder layers        : {cfg.decoder_layers}")
print(f"  attention heads       : {cfg.encoder_attention_heads}")
print(f"  model dimension       : {cfg.d_model}")
print(f"  vocabulary            : {cfg.vocab_size:,} (shared source/target)")
print(f"\nComparing like with like -- {cfg.encoder_layers} encoder layers against")
print(f"{cfg.decoder_layers} decoder layers, embeddings excluded -- the decoder")
print(f"carries {dec_layers/enc_layers:.2f}x the parameters of the encoder.")
print("That ratio is the cross-attention: each decoder layer has an EXTRA")
print("attention block reading the encoder's output, on top of the self-attention")
print("the encoder layers also have. It is the architectural core of translation,")
print("and it is what neither GPT-2 nor DistilBERT has.")

Model : Helsinki-NLP/opus-mt-en-fr
  total parameters      : 75,133,952 (75.1M)
  shared embedding table: 30,471,168 (41%)
  encoder layers only   : 19,176,448 (26%)
  decoder layers only   : 25,486,336 (34%)
  encoder layers        : 6
  decoder layers        : 6
  attention heads       : 8
  model dimension       : 512
  vocabulary            : 59,514 (shared source/target)

Comparing like with like -- 6 encoder layers against
6 decoder layers, embeddings excluded -- the decoder
carries 1.33x the parameters of the encoder.
That ratio is the cross-attention: each decoder layer has an EXTRA
attention block reading the encoder's output, on top of the self-attention
the encoder layers also have. It is the architectural core of translation,
and it is what neither GPT-2 nor DistilBERT has.


In [3]:
SENTENCES = [
    "The warehouse robot delivers the package to the loading bay.",
    "Artificial intelligence is changing how engineers write software.",
    "The cat sat on the mat.",
    "I do not think this approach will work without more data.",
]

def translate(sentences, num_beams=4, max_new_tokens=60):
    batch = mt_tokenizer(sentences, return_tensors="pt", padding=True).to(DEVICE)
    with torch.no_grad():
        generated = mt_model.generate(**batch, num_beams=num_beams,
                                      max_new_tokens=max_new_tokens)
    return mt_tokenizer.batch_decode(generated, skip_special_tokens=True)

translations = translate(SENTENCES)
print("English -> French\n")
for src, tgt in zip(SENTENCES, translations):
    print(f"  EN: {src}")
    print(f"  FR: {tgt}\n")

English -> French

  EN: The warehouse robot delivers the package to the loading bay.
  FR: Le robot d'entrepôt livre le paquet à la baie de chargement.

  EN: Artificial intelligence is changing how engineers write software.
  FR: L'intelligence artificielle change la façon dont les ingénieurs écrivent les logiciels.

  EN: The cat sat on the mat.
  FR: Le chat était assis sur le tapis.

  EN: I do not think this approach will work without more data.
  FR: Je ne pense pas que cette approche fonctionnera sans plus de données.



In [4]:
# How the tokenizer actually splits text -- worth seeing, because it explains
# how a fixed vocabulary copes with words it has never met.
probe = "The warehouse robot delivers the package."
ids = mt_tokenizer(probe)["input_ids"]
pieces = mt_tokenizer.convert_ids_to_tokens(ids)
print(f"Input : {probe}")
print(f"Tokens: {pieces}")
print(f"{len(probe.split())} words -> {len(pieces)} tokens "
      f"(including the </s> end marker)\n")

rare = "The roboticist recalibrated the electromagnetic manipulator."
rare_pieces = mt_tokenizer.convert_ids_to_tokens(mt_tokenizer(rare)["input_ids"])
print(f"Input : {rare}")
print(f"Tokens: {rare_pieces}")
print(f"\nRare words are split into sub-word pieces rather than mapped to an")
print("<UNK> token. This is the direct answer to a problem the Bayesian Networks")
print("lab ran into: there, an unseen word had probability exactly zero and broke")
print("the model. A sub-word vocabulary makes the out-of-vocabulary case")
print("impossible -- anything can be spelled out of smaller pieces.")

Input : The warehouse robot delivers the package.
Tokens: ['▁The', '▁warehouse', '▁robot', '▁delivers', '▁the', '▁package', '.', '</s>']
6 words -> 8 tokens (including the </s> end marker)

Input : The roboticist recalibrated the electromagnetic manipulator.
Tokens: ['▁The', '▁robotic', 'ist', '▁re', 'calibrat', 'ed', '▁the', '▁electromagnetic', '▁man', 'ip', 'ulator', '.', '</s>']

Rare words are split into sub-word pieces rather than mapped to an
<UNK> token. This is the direct answer to a problem the Bayesian Networks
lab ran into: there, an unseen word had probability exactly zero and broke
the model. A sub-word vocabulary makes the out-of-vocabulary case
impossible -- anything can be spelled out of smaller pieces.


In [5]:
# Does the decoding strategy matter? Only where the model is genuinely unsure.
PROBES = [
    ("I do not think this approach will work without more data.", "unambiguous"),
    ("I saw her duck.", "lexical ambiguity: 'duck' is a noun or a verb"),
    ("The old man the boats.", "garden-path: 'man' is the verb"),
]

print("Greedy vs. beam search vs. sampling, on sentences of differing difficulty:\n")
for sentence, why in PROBES:
    batch = mt_tokenizer([sentence], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        greedy = mt_model.generate(**batch, num_beams=1, do_sample=False,
                                   max_new_tokens=40)
        beam = mt_model.generate(**batch, num_beams=5, do_sample=False,
                                 max_new_tokens=40)
        torch.manual_seed(3)
        sampled = mt_model.generate(**batch, do_sample=True, top_k=50,
                                    temperature=1.4, max_new_tokens=40)
    outs = [mt_tokenizer.batch_decode(x, skip_special_tokens=True)[0]
            for x in (greedy, beam, sampled)]
    print(f"  EN     : {sentence}")
    print(f"  ({why})")
    for label, text in zip(("greedy", "beam 5", "sample"), outs):
        print(f"    {label:<7}: {text}")
    print(f"    all three agree: {len(set(outs)) == 1}\n")

print("The result is more interesting than 'beam search is better'.")
print()
print("On the unambiguous sentence all three strategies agree exactly. The model")
print("is confident enough at every step that the argmax path, the best beam and")
print("a random draw all follow the same route -- so the decoding strategy is")
print("irrelevant. Most everyday translation is like this.")
print()
print("'I saw her duck' is where it matters. Greedy produces 'Je l\'ai vue.'")
print("-- 'I saw her' -- having committed early to reading 'duck' as a verb and")
print("then dropped it entirely. Beam search, which scores WHOLE sequences rather")
print("than choosing one token at a time, recovers 'J\'ai vu son canard.'")
print()
print("That is the precise failure greedy decoding has: a token that looks best")
print("locally can leave the model in a state from which no good continuation")
print("exists, and greedy cannot revise it. Beam search keeps several hypotheses")
print("alive long enough for the better one to win. This is the same argument the")
print("Agents lab made about reflex agents -- a locally optimal choice with no")
print("lookahead can be globally wrong -- applied to decoding instead of navigation.")
print()
print("Sampling is the wrong tool here regardless: translation has one intended")
print("meaning, so diversity is a cost rather than a benefit. It is the right tool")
print("for open-ended generation, which is Part 2.")

Greedy vs. beam search vs. sampling, on sentences of differing difficulty:



  EN     : I do not think this approach will work without more data.
  (unambiguous)
    greedy : Je ne pense pas que cette approche fonctionnera sans plus de données.
    beam 5 : Je ne pense pas que cette approche fonctionnera sans plus de données.
    sample : Je ne pense pas que cette approche fonctionnera sans plus de données.
    all three agree: True



  EN     : I saw her duck.
  (lexical ambiguity: 'duck' is a noun or a verb)
    greedy : Je l'ai vue.
    beam 5 : J'ai vu son canard.
    sample : J'ai vu son canard.
    all three agree: False



  EN     : The old man the boats.
  (garden-path: 'man' is the verb)
    greedy : Le vieux les bateaux.
    beam 5 : Le vieil homme les bateaux.
    sample : Le vieil homme les bateaux.
    all three agree: False

The result is more interesting than 'beam search is better'.

On the unambiguous sentence all three strategies agree exactly. The model
is confident enough at every step that the argmax path, the best beam and
a random draw all follow the same route -- so the decoding strategy is
irrelevant. Most everyday translation is like this.

'I saw her duck' is where it matters. Greedy produces 'Je l'ai vue.'
-- 'I saw her' -- having committed early to reading 'duck' as a verb and
then dropped it entirely. Beam search, which scores WHOLE sequences rather
than choosing one token at a time, recovers 'J'ai vu son canard.'

That is the precise failure greedy decoding has: a token that looks best
locally can leave the model in a state from which no good continuation
exists, and greedy canno

---

## Part 2 — Next-token prediction with a decoder-only model

### What GPT-2 actually is

GPT-2 is a **decoder-only** transformer: a stack of causally-masked
self-attention layers ending in a linear projection to vocabulary logits. There
is no encoder and no cross-attention, because there is no second sequence to
attend to — the model's input and output are the same stream of text.

It computes exactly the quantity the Bayesian Networks lab estimated by counting:

$$P(x_t \mid x_1, \dots, x_{t-1})$$

The differences are in the estimator, not the target. A first-order count model
conditions on one previous token and stores each context's distribution
separately. GPT-2 conditions on up to 1024 previous tokens and computes the
distribution through shared weights — which is why an unseen context still
receives a sensible answer instead of a zero.

In [6]:
LM_NAME = "gpt2"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm_model = AutoModelForCausalLM.from_pretrained(LM_NAME).to(DEVICE).eval()
lm_tokenizer.pad_token = lm_tokenizer.eos_token

lcfg = lm_model.config
lm_total = sum(p.numel() for p in lm_model.parameters())
embed = lm_model.transformer.wte.weight.numel()
pos = lm_model.transformer.wpe.weight.numel()
blocks = sum(p.numel() for p in lm_model.transformer.h.parameters())

print(f"Model : {LM_NAME}")
print(f"  total parameters   : {lm_total:,} ({human(lm_total)})")
print(f"  token embeddings   : {embed:,} ({100*embed/lm_total:.0f}%)")
print(f"  position embeddings: {pos:,} ({100*pos/lm_total:.0f}%)")
print(f"  transformer blocks : {blocks:,} ({100*blocks/lm_total:.0f}%)")
print(f"  layers             : {lcfg.n_layer}")
print(f"  attention heads    : {lcfg.n_head}")
print(f"  model dimension    : {lcfg.n_embd}")
print(f"  context window     : {lcfg.n_positions:,} tokens")
print(f"  vocabulary         : {lcfg.vocab_size:,}")

tied = lm_model.lm_head.weight.data_ptr() == lm_model.transformer.wte.weight.data_ptr()
print(f"\n  output head tied to input embedding: {tied}")
print("  Weight tying: the same V x d matrix maps tokens in and logits out. It")
print("  halves the cost of the largest tensor in the model and encodes the")
print("  assumption that a token's input and output representations should match.")

Model : gpt2
  total parameters   : 124,439,808 (124.4M)
  token embeddings   : 38,597,376 (31%)
  position embeddings: 786,432 (1%)
  transformer blocks : 85,054,464 (68%)
  layers             : 12
  attention heads    : 12
  model dimension    : 768
  context window     : 1,024 tokens
  vocabulary         : 50,257

  output head tied to input embedding: True
  Weight tying: the same V x d matrix maps tokens in and logits out. It
  halves the cost of the largest tensor in the model and encodes the
  assumption that a token's input and output representations should match.


In [7]:
# The decoder is CAUSALLY MASKED. Rather than assert that, test it: a token's
# prediction must not change when tokens AFTER it are altered.
prompt = "The warehouse robot moved toward the"
ids = lm_tokenizer(prompt, return_tensors="pt").input_ids

extended_a = torch.cat([ids, lm_tokenizer(" loading bay quickly",
                                          return_tensors="pt").input_ids], dim=1)
extended_b = torch.cat([ids, lm_tokenizer(" dispatch area slowly",
                                          return_tensors="pt").input_ids], dim=1)
with torch.no_grad():
    la = lm_model(extended_a).logits
    lb = lm_model(extended_b).logits

n = ids.shape[1]
same = torch.allclose(la[0, :n], lb[0, :n], atol=1e-5)
diff = float((la[0, :n] - lb[0, :n]).abs().max())
print("Causal-masking check")
print(f"  Two continuations of the same prefix, differing only AFTER position {n}.")
print(f"  Max difference in the logits at positions 0..{n-1}: {diff:.2e}")
print(f"  Identical: {same}")
assert same, "a causal decoder must not let later tokens affect earlier positions"
print("\n  Confirmed: position t's prediction depends only on positions <= t.")
print("  This is what makes the model AUTOREGRESSIVE rather than merely a")
print("  function of the whole string, and it is why one forward pass can score")
print("  every position at once during training while generation must still be")
print("  done one token at a time.")

Causal-masking check
  Two continuations of the same prefix, differing only AFTER position 6.
  Max difference in the logits at positions 0..5: 0.00e+00
  Identical: True

  Confirmed: position t's prediction depends only on positions <= t.
  This is what makes the model AUTOREGRESSIVE rather than merely a
  function of the whole string, and it is why one forward pass can score
  every position at once during training while generation must still be
  done one token at a time.


In [8]:
def next_token_distribution(prompt, k=10):
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = lm_model(ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(lm_tokenizer.decode([i]), float(p))
            for i, p in zip(top.indices, top.values)], probs

print("Inspecting P(next token | context) directly -- the same object the")
print("Bayesian Networks lab built by counting:\n")
for prompt in ["The warehouse robot delivers the",
               "The capital of France is",
               "2 + 2 ="]:
    top, probs = next_token_distribution(prompt)
    entropy = float(-(probs * torch.log(probs + 1e-12)).sum())
    print(f'  "{prompt}"')
    print(f"    entropy {entropy:.2f} nats over {len(probs):,} tokens "
          f"(uniform would be {torch.log(torch.tensor(float(len(probs)))):.2f})")
    for tok, p in top[:6]:
        bar = "#" * round(p * 40)
        print(f"      {tok!r:<14} {p:.4f}  {bar}")
    print(f"    top-10 mass: {sum(p for _, p in top):.4f}")
    print()
print("Entropy is the useful number. A low-entropy distribution means the model")
print("is confident (few plausible continuations); a high-entropy one means the")
print("context genuinely underdetermines what comes next. The softmax sums to 1")
print("over all 50,257 tokens, exactly as in the three-class experiment from the")
print("Neural Models lab -- only the number of classes has changed.")

Inspecting P(next token | context) directly -- the same object the
Bayesian Networks lab built by counting:

  "The warehouse robot delivers the"
    entropy 7.01 nats over 50,257 tokens (uniform would be 10.82)
      ' goods'       0.0772  ###
      ' food'        0.0317  #
      ' robot'       0.0210  #
      ' first'       0.0195  #
      ' items'       0.0160  #
      ' supplies'    0.0136  #
    top-10 mass: 0.2243

  "The capital of France is"
    entropy 6.00 nats over 50,257 tokens (uniform would be 10.82)
      ' the'         0.0846  ###
      ' now'         0.0479  ##
      ' a'           0.0462  ##
      ' France'      0.0324  #
      ' Paris'       0.0322  #
      ' in'          0.0266  #
    top-10 mass: 0.3591

  "2 + 2 ="
    entropy 5.12 nats over 50,257 tokens (uniform would be 10.82)
      ' 3'           0.1020  ####
      ' 1'           0.0916  ####
      ' 2'           0.0842  ###
      ' 0'           0.0752  ###
      ' 4'           0.0550  ##
      ' 5'           

In [9]:
START_SENTENCE = "In the future, artificial intelligence will"

print(f'Prompt: "{START_SENTENCE}"\n')
inputs = lm_tokenizer(START_SENTENCE, return_tensors="pt")

configs = [
    ("greedy (argmax)", dict(do_sample=False, num_beams=1)),
    ("beam search (4)", dict(do_sample=False, num_beams=4)),
    ("sampling, T=0.7", dict(do_sample=True, temperature=0.7, top_k=50)),
    ("sampling, T=1.3", dict(do_sample=True, temperature=1.3, top_k=50)),
]
for label, kwargs in configs:
    torch.manual_seed(407)
    with torch.no_grad():
        out = lm_model.generate(**inputs, max_new_tokens=40,
                                pad_token_id=lm_tokenizer.eos_token_id, **kwargs)
    text = lm_tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"  [{label}]")
    print(wrap(text))
    print()

Prompt: "In the future, artificial intelligence will"



  [greedy (argmax)]
    In the future, artificial intelligence will be able to do things like
    search for information about people, and to do things like search for
    information about people.  "We're going to see a lot of things that are
    going to be



  [beam search (4)]
    In the future, artificial intelligence will be able to do things that
    humans can't do.  In the future, artificial intelligence will be able to
    do things that humans can't do. In the future, artificial intelligence
    will be able



  [sampling, T=0.7]
    In the future, artificial intelligence will become used to better
    understand the world around us. This is an important step in that
    direction, because the more we understand, the more we can use it to
    improve the lives of other people.



  [sampling, T=1.3]
    In the future, artificial intelligence will only accelerate from humans to
    computer chips in 20 to 50 years time limit. It requires hundreds, tens of
    thousands of years to overcome complex technological hurdles.  The United
    Nations, for example, says



In [10]:
# Quantify the temperature effect rather than eyeballing it.
print("Temperature reshapes the SAME distribution before sampling:\n")
ids = lm_tokenizer("The warehouse robot delivers the", return_tensors="pt").input_ids
with torch.no_grad():
    logits = lm_model(ids).logits[0, -1]

print("| temperature | entropy (nats) | top-1 prob | tokens covering 90% mass |")
print("|---:|---:|---:|---:|")
for T in (0.3, 0.7, 1.0, 1.5, 2.0):
    p = torch.softmax(logits / T, dim=-1)
    ent = float(-(p * torch.log(p + 1e-12)).sum())
    srt = torch.sort(p, descending=True).values
    n90 = int((torch.cumsum(srt, 0) < 0.9).sum()) + 1
    print(f"| {T} | {ent:.3f} | {float(srt[0]):.4f} | {n90:,} |")

print("\nLow temperature sharpens the distribution toward its argmax (entropy")
print("falls, top-1 probability rises); high temperature flattens it toward")
print("uniform. T -> 0 is greedy decoding; T -> inf is uniform sampling.")
print("Note that temperature changes NOTHING about the model -- the logits are")
print("fixed. It is purely a decoding-time reshaping, which is why it can be")
print("tuned per request without retraining.")

Temperature reshapes the SAME distribution before sampling:

| temperature | entropy (nats) | top-1 prob | tokens covering 90% mass |
|---:|---:|---:|---:|
| 0.3 | 0.434 | 0.9144 | 1 |
| 0.7 | 4.165 | 0.2985 | 265 |
| 1.0 | 7.010 | 0.0772 | 3,155 |
| 1.5 | 8.976 | 0.0119 | 11,947 |
| 2.0 | 9.693 | 0.0034 | 18,759 |

Low temperature sharpens the distribution toward its argmax (entropy
falls, top-1 probability rises); high temperature flattens it toward
uniform. T -> 0 is greedy decoding; T -> inf is uniform sampling.
Note that temperature changes NOTHING about the model -- the logits are
fixed. It is purely a decoding-time reshaping, which is why it can be
tuned per request without retraining.


---

## Part 3 — Sentiment analysis with an encoder-only model

### A different shape of problem

Sentiment analysis is **sequence classification**: many tokens in, one label out.
That calls for a third architecture.

An encoder-only model (BERT and its relatives) uses **bidirectional**
self-attention — every token attends to every other in both directions. For
classification that is exactly right and it is also *safe*, because there is no
generation step and therefore no risk of leaking future information: the whole
input is legitimately available at once. A causal model like GPT-2 would have to
read the sentence left to right, so its representation of the first word could
not account for the last — a real handicap when the sentiment turns on a word at
the end ("...which was a complete disaster").

The model here is DistilBERT — a distilled, 6-layer BERT — fine-tuned on SST-2,
a binary movie-review sentiment corpus. Two pieces are worth separating: the
pretrained encoder (general language representation, learned by masked language
modelling) and the small classification head bolted on top (2 output units,
trained on SST-2). This is the transfer-learning pattern, and the parameter
counts below show how lopsided it is.

In [11]:
CLS_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
cls_tokenizer = AutoTokenizer.from_pretrained(CLS_NAME)
cls_model = AutoModelForSequenceClassification.from_pretrained(CLS_NAME).to(DEVICE).eval()

ccfg = cls_model.config
cls_total = sum(p.numel() for p in cls_model.parameters())
backbone = sum(p.numel() for p in cls_model.distilbert.parameters())
head = cls_total - backbone

print(f"Model : {CLS_NAME}")
print(f"  total parameters     : {cls_total:,} ({human(cls_total)})")
print(f"  pretrained encoder   : {backbone:,} ({100*backbone/cls_total:.1f}%)")
print(f"  classification head  : {head:,} ({100*head/cls_total:.1f}%)")
print(f"  layers               : {ccfg.n_layers}")
print(f"  attention heads      : {ccfg.n_heads}")
print(f"  model dimension      : {ccfg.dim}")
print(f"  labels               : {ccfg.id2label}")
print(f"\nThe head is {100*head/cls_total:.1f}% of the model. Essentially all the")
print("capability is in the pretrained encoder; fine-tuning on SST-2 adapted")
print("those weights and trained a tiny classifier on top. That asymmetry is the")
print("whole economic argument for pretraining.")

Model : distilbert-base-uncased-finetuned-sst-2-english
  total parameters     : 66,955,010 (67.0M)
  pretrained encoder   : 66,362,880 (99.1%)
  classification head  : 592,130 (0.9%)
  layers               : 6
  attention heads      : 12
  model dimension      : 768
  labels               : {0: 'NEGATIVE', 1: 'POSITIVE'}

The head is 0.9% of the model. Essentially all the
capability is in the pretrained encoder; fine-tuning on SST-2 adapted
those weights and trained a tiny classifier on top. That asymmetry is the
whole economic argument for pretraining.


In [12]:
REVIEWS = [
    "This laboratory was genuinely interesting and I learned a lot.",
    "The instructions were confusing and the code did not run.",
    "The film was long.",
    "It is not bad at all.",
    "I wanted to love this, but it fell apart in the second half.",
    "A masterpiece of tedium.",
]

def classify(texts):
    batch = cls_tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        logits = cls_model(**batch).logits
    probs = torch.softmax(logits, dim=-1)
    out = []
    for text, row in zip(texts, probs):
        idx = int(row.argmax())
        out.append((text, ccfg.id2label[idx], float(row[idx]), row))
    return out

print("| text | label | confidence | P(NEG) | P(POS) |")
print("|---|:---:|---:|---:|---:|")
results = classify(REVIEWS)
for text, label, conf, row in results:
    short = text if len(text) < 52 else text[:49] + "..."
    print(f"| {short} | **{label}** | {conf:.4f} | {float(row[0]):.4f} | {float(row[1]):.4f} |")

for _, _, _, row in results:
    assert abs(float(row.sum()) - 1.0) < 1e-5
print("\nEvery row sums to 1 -- the same softmax + cross-entropy output pairing as")
print("the three-class experiment in the Neural Models lab, with K = 2 here.")

| text | label | confidence | P(NEG) | P(POS) |
|---|:---:|---:|---:|---:|


| This laboratory was genuinely interesting and I l... | **POSITIVE** | 0.9989 | 0.0011 | 0.9989 |
| The instructions were confusing and the code did ... | **NEGATIVE** | 0.9980 | 0.9980 | 0.0020 |
| The film was long. | **NEGATIVE** | 0.9965 | 0.9965 | 0.0035 |
| It is not bad at all. | **POSITIVE** | 0.9996 | 0.0004 | 0.9996 |
| I wanted to love this, but it fell apart in the s... | **NEGATIVE** | 0.9996 | 0.9996 | 0.0004 |
| A masterpiece of tedium. | **NEGATIVE** | 0.9863 | 0.9863 | 0.0137 |

Every row sums to 1 -- the same softmax + cross-entropy output pairing as
the three-class experiment in the Neural Models lab, with K = 2 here.


### Where the model is right for the wrong reasons

The last four examples were chosen to probe rather than to flatter, and they
expose the difference between *classifying sentiment* and *matching sentiment
vocabulary*:

- **"The film was long."** carries no sentiment word at all. Any confident answer
  here is the model importing a prior from its training distribution — in movie
  reviews, "long" skews negative — rather than reading the sentence.
- **"It is not bad at all."** requires composing a negation with a negative word
  to produce a positive. A bag-of-words model gets this wrong reliably; a
  bidirectional transformer can in principle get it right, because "not" and
  "bad" attend to each other.
- **"I wanted to love this, but it fell apart..."** contains strong positive
  vocabulary in a clause that the sentence then overrides. The correct reading
  depends on discourse structure, not word counts.
- **"A masterpiece of tedium."** is sarcasm: every word is positive except the
  one that inverts it.

The confidences are the interesting part. A model that is *confidently wrong* on
these is more concerning than one that is uncertain, because calibration is what
makes a classifier usable downstream.

In [13]:
print("Confidence on the deliberately hard cases:\n")
hard = [
    ("The film was long.", "no sentiment word at all"),
    ("It is not bad at all.", "negation of a negative -> positive"),
    ("I wanted to love this, but it fell apart in the second half.", "contrastive clause"),
    ("A masterpiece of tedium.", "sarcasm"),
]
for (text, why), (_, label, conf, _) in zip(hard, classify([h[0] for h in hard])):
    flag = "HIGH CONFIDENCE" if conf > 0.95 else "hedged"
    print(f'  "{text}"')
    print(f"    challenge : {why}")
    print(f"    verdict   : {label} at {conf:.4f}  [{flag}]\n")

print("The model is binary by construction -- SST-2 has only POSITIVE and")
print("NEGATIVE -- so it CANNOT express 'neutral' or 'uncertain' as a label. A")
print("genuinely neutral sentence must still be forced into one of two classes,")
print("and softmax will still report a probability for it. This is a property of")
print("the label set, not a failure of the network, and it is the kind of thing")
print("that has to be checked before deploying a classifier: the output space")
print("must actually contain the answers you need.")

Confidence on the deliberately hard cases:

  "The film was long."
    challenge : no sentiment word at all
    verdict   : NEGATIVE at 0.9965  [HIGH CONFIDENCE]

  "It is not bad at all."
    challenge : negation of a negative -> positive
    verdict   : POSITIVE at 0.9996  [HIGH CONFIDENCE]

  "I wanted to love this, but it fell apart in the second half."
    challenge : contrastive clause
    verdict   : NEGATIVE at 0.9996  [HIGH CONFIDENCE]

  "A masterpiece of tedium."
    challenge : sarcasm
    verdict   : NEGATIVE at 0.9863  [HIGH CONFIDENCE]

The model is binary by construction -- SST-2 has only POSITIVE and
NEGATIVE -- so it CANNOT express 'neutral' or 'uncertain' as a label. A
genuinely neutral sentence must still be forced into one of two classes,
and softmax will still report a probability for it. This is a property of
the label set, not a failure of the network, and it is the kind of thing
that has to be checked before deploying a classifier: the output space
must actuall

---

## The three architectures compared

Everything in this notebook is a transformer. The differences come down to
**what attends to what**, and that single choice determines what each is for.

In [14]:
print("| | MarianMT (enc-dec) | GPT-2 (dec-only) | DistilBERT (enc-only) |")
print("|---|---|---|---|")
print(f"| task shape | seq -> seq | seq -> next token | seq -> label |")
print(f"| encoder attention | bidirectional | - | bidirectional |")
print(f"| decoder attention | causal | causal | - |")
print(f"| cross-attention | **yes** | no | no |")
print(f"| parameters | {human(total)} | {human(lm_total)} | {human(cls_total)} |")
print(f"| layers | {cfg.encoder_layers}+{cfg.decoder_layers} | {lcfg.n_layer} | {ccfg.n_layers} |")
print(f"| vocabulary | {cfg.vocab_size:,} | {lcfg.vocab_size:,} | {ccfg.vocab_size:,} |")
print(f"| output | token distribution | token distribution | 2 class logits |")
print(f"| trained by | parallel-corpus seq2seq | next-token prediction | MLM, then SST-2 |")

print("\nThree rules that follow from the attention pattern:\n")
print("  1. Need to GENERATE text? The decoder must be causally masked, because")
print("     it cannot attend to tokens it has not produced yet. (Verified above.)")
print("  2. Need to CONDITION on a separate input sequence? You need")
print("     cross-attention, i.e. an encoder-decoder. GPT-2 has no mechanism for")
print("     a second sequence -- prompting works by concatenating into ONE stream.")
print("  3. Only need to READ? Drop the decoder. Bidirectional attention is")
print("     strictly more informative and there is no leakage risk without")
print("     generation.")
print("\nThe modern twist is that rule 2 has been partly overturned in practice:")
print("large decoder-only models do translation well by putting both sentences in")
print("one stream, trading the architectural bias of cross-attention for scale.")
print("That is an empirical result about capacity, not a refutation of the")
print("architecture -- a 75M-parameter model like MarianMT still benefits from")
print("having the alignment structure built in rather than learned.")

| | MarianMT (enc-dec) | GPT-2 (dec-only) | DistilBERT (enc-only) |
|---|---|---|---|
| task shape | seq -> seq | seq -> next token | seq -> label |
| encoder attention | bidirectional | - | bidirectional |
| decoder attention | causal | causal | - |
| cross-attention | **yes** | no | no |
| parameters | 75.1M | 124.4M | 67.0M |
| layers | 6+6 | 12 | 6 |
| vocabulary | 59,514 | 50,257 | 30,522 |
| output | token distribution | token distribution | 2 class logits |
| trained by | parallel-corpus seq2seq | next-token prediction | MLM, then SST-2 |

Three rules that follow from the attention pattern:

  1. Need to GENERATE text? The decoder must be causally masked, because
     it cannot attend to tokens it has not produced yet. (Verified above.)
  2. Need to CONDITION on a separate input sequence? You need
     cross-attention, i.e. an encoder-decoder. GPT-2 has no mechanism for
     a second sequence -- prompting works by concatenating into ONE stream.
  3. Only need to READ? Drop the d

---

## Connecting back: the same conditional, three times

This course has now built $P(x_t \mid x_{<t})$ three times.

| | Bayesian Networks lab | Neural Models lab | This lab (GPT-2) |
|---|---|---|---|
| Representation | explicit count table | 9-parameter network | 124M-parameter transformer |
| Context | 1–2 previous tokens | one 2-bit input | up to 1024 tokens |
| Parameters | one per observed pair | 9 | 124,439,808 |
| Learning | counting and normalising | gradient descent | gradient descent |
| Unseen input | probability **exactly 0** | n/a (inputs exhaustive) | a smooth distribution |
| Output layer | normalised counts | softmax over 3 classes | softmax over 50,257 |
| Loss | (none — closed form) | cross-entropy | cross-entropy |
| Logit gradient | n/a | $p - y$ | $p - y$ |

**What is mathematically identical.** The chain-rule factorisation; the softmax
normalisation $\sum_v P(v \mid \text{context}) = 1$; the cross-entropy loss; and
the logit gradient $p - y$, which the Neural Models lab verified numerically on a
three-class problem and which holds unchanged at $V = 50{,}257$. The generation
loop — sample, append, re-condition — is the same three lines.

**What changes.** The estimator, and one property that follows from it:
**parameter sharing**. The count model stores every context's distribution
separately, so nothing learned about one context transfers to a similar one —
which is exactly why its second-order version overfitted and assigned probability
zero to held-out sentences. A transformer maps contexts through shared weights,
so similar contexts give similar predictions and an unseen context still receives
a sensible answer. That is generalisation, and it is the thing the count-based
model structurally cannot do.

The output matrix is also now one of the largest tensors in the model —
$50{,}257 \times 768 \approx 38.6$M parameters, about 31% of GPT-2 — which is why
it is tied to the input embedding, and why the Neural Models lab's remark about
fused cross-entropy kernels and chunked logits matters at this scale.

In [15]:
print("Verifying the shared arithmetic, at GPT-2 scale:\n")
ids = lm_tokenizer("The warehouse robot delivers the", return_tensors="pt").input_ids
with torch.no_grad():
    logits = lm_model(ids).logits[0, -1]
probs = torch.softmax(logits, dim=-1)

print(f"  vocabulary size            : {len(probs):,}")
print(f"  sum of probabilities       : {float(probs.sum()):.10f}")
print(f"  |sum - 1|                  : {abs(float(probs.sum()) - 1):.2e}")
assert abs(float(probs.sum()) - 1) < 1e-4

shifted = torch.softmax(logits + 1000.0, dim=-1)
print(f"  invariant to a +1000 shift : {torch.allclose(probs, shifted, atol=1e-6)}")
print("    (the same max-subtraction stability the Neural Models lab examined,")
print("     here over 50,257 logits instead of 3)")

# dL/dz = p - y, on a real GPT-2 forward pass
target = torch.tensor([lm_tokenizer(" package").input_ids[0]])
live = lm_model(ids).logits[0, -1:]
live.retain_grad()
loss = torch.nn.functional.cross_entropy(live, target)
lm_model.zero_grad()
loss.backward()
with torch.no_grad():
    p = torch.softmax(live, dim=-1)
    onehot = torch.nn.functional.one_hot(target, len(probs)).float()
    expected = p - onehot
err = float((live.grad - expected).abs().max())
print(f"\n  dL/dz == (p - y)           : max difference {err:.2e} "
      f"-> {err < 1e-6}")
assert err < 1e-6
print("    The identity derived by hand on a 3-class toy holds exactly on a")
print("    124M-parameter model with 50,257 classes. The mathematics did not")
print("    change; only the function computing the logits did.")

Verifying the shared arithmetic, at GPT-2 scale:

  vocabulary size            : 50,257
  sum of probabilities       : 1.0000220537
  |sum - 1|                  : 2.21e-05
  invariant to a +1000 shift : True
    (the same max-subtraction stability the Neural Models lab examined,
     here over 50,257 logits instead of 3)



  dL/dz == (p - y)           : max difference 7.45e-09 -> True
    The identity derived by hand on a 3-class toy holds exactly on a
    124M-parameter model with 50,257 classes. The mathematics did not
    change; only the function computing the logits did.


---

## Consolidated results

In [16]:
print("=" * 78)
print("RESULT SUMMARY".center(78))
print("=" * 78)

print(f"\n[1] Translation -- {MT_NAME}")
print(f"    encoder-decoder, {cfg.encoder_layers}+{cfg.decoder_layers} layers, "
      f"{human(total)} parameters")
print(f"    decoder layers carry {dec_layers/enc_layers:.2f}x the encoder parameters")
print(f"    (embeddings excluded; the difference is the cross-attention block)")
print(f"    translated {len(SENTENCES)} sentences EN->FR")
print(f"    greedy vs beam identical on easy input; diverge on ambiguous input")
print(f"    sub-word tokenisation means no out-of-vocabulary case exists")

print(f"\n[2] Next-token prediction -- {LM_NAME}")
print(f"    decoder-only, {lcfg.n_layer} layers, {human(lm_total)} parameters, "
      f"{lcfg.n_positions}-token context")
print(f"    causal masking VERIFIED: future tokens do not affect earlier positions")
print(f"    output head tied to input embedding: {tied}")
print(f"    softmax over {lcfg.vocab_size:,} tokens sums to 1; shift-invariant")
print(f"    dL/dz == (p - y) confirmed numerically on a real forward pass")
print(f"    temperature sweep: entropy and top-1 mass measured at T = 0.3 .. 2.0")

print(f"\n[3] Sentiment analysis -- {CLS_NAME}")
print(f"    encoder-only, {ccfg.n_layers} layers, {human(cls_total)} parameters")
print(f"    classification head is only {100*head/cls_total:.1f}% of the model")
print(f"    tested on negation, sarcasm and contrastive clauses, not just easy cases")
print(f"    label set is binary by construction: 'neutral' is not expressible")

print(f"\n[Cross-cutting] the same P(x_t | x_<t) as the BN and Neural Models labs;")
print(f"    identical factorisation, softmax normalisation and p - y gradient.")
print(f"    The estimator changes; the probabilistic question does not.")
print("\n" + "=" * 78)

                                RESULT SUMMARY                                

[1] Translation -- Helsinki-NLP/opus-mt-en-fr
    encoder-decoder, 6+6 layers, 75.1M parameters
    decoder layers carry 1.33x the encoder parameters
    (embeddings excluded; the difference is the cross-attention block)
    translated 4 sentences EN->FR
    greedy vs beam identical on easy input; diverge on ambiguous input
    sub-word tokenisation means no out-of-vocabulary case exists

[2] Next-token prediction -- gpt2
    decoder-only, 12 layers, 124.4M parameters, 1024-token context
    causal masking VERIFIED: future tokens do not affect earlier positions
    output head tied to input embedding: True
    softmax over 50,257 tokens sums to 1; shift-invariant
    dL/dz == (p - y) confirmed numerically on a real forward pass
    temperature sweep: entropy and top-1 mass measured at T = 0.3 .. 2.0

[3] Sentiment analysis -- distilbert-base-uncased-finetuned-sst-2-english
    encoder-only, 6 layers, 67.0M 

---

## Reflection

### What using pretrained models actually taught me

**The architecture is a claim about the task, not a detail.** Before this lab I
would have described all three as "transformers". They are, but the choice
between them is forced by the shape of the problem: generation requires causal
masking; conditioning on a separate sequence requires cross-attention;
classification wants bidirectional attention and no decoder at all. The
parameter-count breakdown made this concrete — the MarianMT decoder is heavier
than its encoder at equal depth precisely because each decoder layer carries an
extra attention block, and that block is the part that does the translating.

**Verification still applies, and it is cheap.** I could not retrain these models
or inspect their training data, but I could still test structural properties.
Checking that the softmax sums to 1 over 50,257 tokens, that it is invariant to a
constant logit shift, that $\partial L/\partial z = p - y$, and that the causal
mask genuinely blocks future positions — all of these are assertions that would
fail loudly if my mental model of the architecture were wrong. Using a model
someone else trained does not exempt you from checking what kind of object it is.

**The hard part of evaluation is choosing the inputs.** The sentiment classifier
looks flawless on "this was great" and "this was terrible", and those examples
teach nothing. The examples worth running are the ones that separate *classifying
sentiment* from *matching sentiment vocabulary*: negation, sarcasm, contrastive
clauses, and sentences with no sentiment vocabulary at all. This is the same
principle as the Search lab's unreachable-goal test and the Logic lab's
deliberately-broken planner — a test only carries information if it could have
failed.

**Confidence is not accuracy.** A binary classifier will report a probability for
a neutral sentence because its label set contains no other option. The output
space has to be checked against the question being asked before the numbers mean
anything, and this is exactly the "output layer and loss must be chosen together"
point from the Neural Models lab, arriving from the other direction.

### Limitations of what is shown here

These are small models — 74M, 124M and 67M parameters — and their behaviour is
not representative of current large models. GPT-2's generations wander and repeat
in ways that a modern model does not, and the translations are competent but not
idiomatic. Nothing here was measured against a benchmark; the outputs are
illustrations, not evaluations. A real assessment of translation quality needs a
held-out parallel test set and a metric (BLEU, COMET), and a real assessment of
the classifier needs a labelled test set with a confusion matrix rather than six
hand-picked sentences.

---

## Reference

S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed.
A. Vaswani et al., *Attention Is All You Need*, NeurIPS 2017 — the encoder–decoder
transformer, and the attention patterns that distinguish the three families used
here.